In [ ]:
import pandas as pd
import numpy as np
import re
import plotly.graph_objects as go
import plotly.express as px
import os

In [31]:
date="02_02_26.csv"
df=pd.read_csv(fr"C:\Users\Administrator\Desktop\ds\f4e\githunguri1\Githunguri-kitchen-f4e-dashboard\data\data_raw\extarcted_csv\{date}")
print(df.shape)

(75, 27)


In [ ]:
df.head(2)

In [ ]:
df.isnull().sum()

In [ ]:
#stem and map the school names
df["School"]

In [ ]:
def clean_school_name(name):
    # Fixing  this typos for mathanja 
    name = name.replace("Primry", "Primary")
    
    # Replacing "Comprehensive" with "Primary"
    name = name.replace("Comprehensive", "Primary")
    
    # chopping the word "School" 
    name = re.sub(r"\b[Ss]chool\b", "", name)
    
    # remaoving  Junior Secondary" or similar suffixes
    name = re.sub(r"& Junior Secondary", "", name, flags=re.IGNORECASE)
    
    # Simplifying Catholic Archdiocese   and pcea
    if "Gataka" in name:
        return "Gataka Primary"
    if "Mitahato" in name:
        if "Secondary" in name:
            return "Mitahato Secondary"
        else:
            return "Mitahato Primary"
    if "Matuguta" in name:
        if "Secondary" in name:
            return "Matuguta Secondary"
        else:
            return "Matuguta Primary"
    if "Riamute" in name:
        return "Riamute Primary"
    if "Ngemwa" in name:
        if "Secondary" in name:
            return "Ngemwa Secondary"
        else:
            return "Ngemwa Primary"
    if "Lioki" in name:
        if "Secondary" in name:
            return "Lioki Secondary"
        else:
            return "Lioki Primary"
    if "Karia" in name:
        if "Secondary" in name:
            return "Karia Secondary"
        else:
            return "Karia Primary"
    if "kamondo" in name:
        if "Secondary" in name:
            return "Kamondo Secondary"
        else:
            return "Kamondo Primary"
    
    
    #    spacing and capitalization
    name = " ".join(name.split())  # remove extra spaces
    name = name.title()            # standardize case
    
    return name.strip()



In [35]:
df["School"] = df["School"].apply(clean_school_name)


In [34]:
pd.describe_option("display.max_rows")
pd.set_option("display.max_rows", 100)

display.max_rows : int
    If max_rows is exceeded, switch to truncate view. Depending on
    `large_repr`, objects are either centrally truncated or printed as
    a summary view. 'None' value means unlimited.

    In case python/IPython is running in a terminal and `large_repr`
    equals 'truncate' this can be set to 0 and pandas will auto-detect
    the height of the terminal and print a truncated object which fits
    the screen height. The IPython notebook, IPython qtconsole, or
    IDLE do not run in a terminal and hence it is not possible to do
    correct auto-detection.
    [default: 60] [currently: 100]


In [36]:
df['School']

0             William Ng'Iru Secondary
1      Kagema Primary Date: 02/02/2026
2                     Kanjai Secondary
3                     Miguta Secondary
4                      Gathugu Primary
5                      Mukua Secondary
6           Gikang'A Kageche Secondary
7                     Matuguta Primary
8                       Gataka Primary
9                        Miiri Primary
10                      Kambui Primary
11                  Gathangari Primary
12                    Githioro Primary
13                   Gathaithi Primary
14                     Githiga Primary
15                    Gitiha Secondary
16                Kiambururu Secondary
17                  Matuguta Secondary
18                      Gatitu Primary
19                     Kanyore Primary
20                       Ikinu Primary
21                  Gathiruini Primary
22                  Kiambururu Primary
23                      Miguta Primary
24                 Gathaithi Secondary
25                     Nd

In [ ]:

# creating a new columns of  the kichen served for each school 

kiambu_kitchen_schools=["Githunguri Technical & Secondary", "Kahunira Primary", "Ciiko Primary", "Kahunira Secondary", "Kiababu Primary"]
def map_kitchen_served(school_name):
    if school_name in kiambu_kitchen_schools:
        return 'kiambu kitchen'
    else:
        return 'Githunguri'
df["Kitchen"] = df["School"].apply(map_kitchen_served)

In [ ]:
df.head()

In [ ]:
#schools type 
def get_school_type(name):
    name = name.lower()
    if "secondary" in name:
        return "Secondary"
    else:
        return "Primary"

df["School_Type"] = df["School"].apply(get_school_type)

#for good visual  i rearrabge the columns so school type can be be next to school name
cols = list(df.columns)
cols.insert(2, cols.pop(cols.index("School_Type")))
df = df[cols]




In [ ]:
df.head(30)

In [ ]:
df.head(2)

In [ ]:
def get_rice_variant(menu_item):
    if pd.isna(menu_item):
        return ""
    menu_item = menu_item.lower()
    if "masala" in menu_item:
        return "Masala"
    elif "tumeric" in menu_item:
        return "Tumeric"
    else:
        return "White"

df["Rice_variant"] = df["Menu"].apply(get_rice_variant)
# For column reordering to be next to menu 
cols = list(df.columns)
cols.insert(5, cols.pop(cols.index("Rice_variant")))
df = df[cols]



In [ ]:
def clean_menu(name):
    if pd.isna(name):
        return None
    
    # Normalize casing and whitespace
    name = name.strip().title()
    
    # Replace connectors
    name = name.replace("&", "And")
    
    # Standardize synonyms/abbreviations
    name = re.sub(r"\bGg\b", "Greengrams", name, flags=re.IGNORECASE)
    name = name.replace("Green Gram", "Greengrams")
    name = name.replace("Green Grams", "Greengrams")
    
    # If "Rice" exists, discard everything before it
    if "Rice" in name:
        idx = name.find("Rice")
        name = name[idx:]
    
    # Canonical mapping
    if "Greengrams" in name:
        return "Rice and Greengrams"
    elif "Beans" in name:
        return "Rice and Beans"
    elif "Western" in name:
        return "Rice and Western"
    
    # If no keyword matched, return substring after Rice
    return name

In [ ]:
df["Menu"] = df["Menu"].apply(clean_menu)

In [ ]:
df.info()

In [ ]:
# Convert to datetime
df["Start_Taping"] = pd.to_datetime(df["Start_Taping"], format="%d %b %Y, %H:%M:%S")
df["Stop_Taping"] = pd.to_datetime(df["Stop_Taping"], format="%d %b %Y, %H:%M:%S")
# Compute duration
lunch_time= df["Stop_Taping"] - df["Start_Taping"]
df["Lunch_Time_Min"] = (lunch_time.dt.total_seconds() / 60).round(0).astype('Int64')  # Convert to minutes and round


cols = list(df.columns)
# moving Lunch_Time_Min to index 23 nest to the parent columns
cols.insert(23, cols.pop(cols.index("Lunch_Time_Min")))

# Reassign DataFrame with new order
df = df[cols]



In [ ]:

pd.set_option("display.max_rows", None)
df.head(3)

In [ ]:
df.head(10)

In [ ]:
df.nunique()

In [ ]:
#lets use os 
output_dir = r"C:\Users\Administrator\Desktop\ds\f4e\githunguri1\Githunguri-kitchen-f4e-dashboard\data\data_processed"
base = date.replace(".csv", "")
output_path = os.path.join(output_dir, f"{base}_clean.csv")
df.to_csv(output_path, index=False)
print("Saved:", output_path)